# Lab 02.2 — Formati di file e Apache Parquet

## Introduzione
Spark legge e scrive molti formati con la stessa API (`spark.read.format(...)`, `df.write.format(...)`).
I formati si dividono in due famiglie:

| | Orientati alle **righe** | Orientati alle **colonne** |
|---|---|---|
| Esempi | CSV, JSON, Avro | **Parquet**, ORC |
| Schema | assente (CSV) o ripetuto (JSON) | salvato nei metadati |
| Lettura di poche colonne | legge tutto il file | legge solo le colonne utili |
| Compressione | scarsa | ottima (valori simili vicini) |

Parquet organizza i dati in **row group** → **column chunk** → **pagine**, e memorizza per ogni chunk statistiche (min/max, null) che permettono di **saltare** blocchi interi.

### Obiettivi
1. Inizializzare la sessione e preparare il dataset degli ordini.
2. Scrivere lo stesso DataFrame in CSV, JSON, ORC e Parquet confrontando tempi e dimensioni.
3. Confrontare lo schema inferito (CSV) con quello memorizzato (Parquet).
4. Misurare il vantaggio della lettura colonnare (*column pruning*).
5. Esplorare i metadati di un file Parquet con PyArrow.
6. Osservare il *predicate pushdown* e il *partition pruning* (`partitionBy`).
7. Confrontare i codec di compressione ed evolvere lo schema (`mergeSchema`).
8. Usare SQL direttamente sui file e tabelle del catalogo.

## Step 1: Inizializzare la SparkSession e preparare i dati

Oltre alla sessione definiamo:
* `spark.sql.warehouse.dir`: cartella in cui il catalogo salva le tabelle gestite (serve nello Step 11);
* `OUT`: cartella di output, svuotata a ogni esecuzione per ripartire da zero;
* `dimensione()`: funzione di utilità che somma i byte dei file scritti da Spark;
* `ordini`: il DataFrame di partenza, messo in `cache()` così da non rigenerarlo a ogni scrittura (`count()` lo materializza).

In [ ]:
import os, time, glob, shutil
from pyspark.sql import SparkSession, functions as F
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-Parquet")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")
         # cartella in cui il catalogo salva le tabelle gestite (sezione 9)
         .config("spark.sql.warehouse.dir", "/home/jovyan/data/warehouse")
         .getOrCreate())

OUT = "/home/jovyan/data/formati"
shutil.rmtree(OUT, ignore_errors=True)

def dimensione(path):
    """Dimensione su disco (MB) di una cartella scritta da Spark."""
    return sum(os.path.getsize(f) for f in glob.glob(f"{path}/**/*", recursive=True) if os.path.isfile(f)) / 1e6

ordini = dati_shop.carica(spark)["ordini"].cache()
print(f"{ordini.count():,} ordini")

## Step 2: Scrivere lo stesso DataFrame in formati diversi

Ogni partizione del DataFrame viene scritta in parallelo da un executor in un file `part-XXXXX`.
Ciclando su CSV, JSON, ORC e Parquet misuriamo tempo di scrittura e spazio occupato: i formati colonnari sono tipicamente più compatti.

In [ ]:
formati = {"csv": {"header": True}, "json": {}, "orc": {}, "parquet": {}}
for fmt, opzioni in formati.items():
    t0 = time.time()
    ordini.write.format(fmt).options(**opzioni).mode("overwrite").save(f"{OUT}/ordini_{fmt}")
    print(f"{fmt:8s} scritto in {time.time() - t0:5.2f}s  →  {dimensione(f'{OUT}/ordini_{fmt}'):7.1f} MB")

### Step 2b: Contenuto della cartella di output

Spark scrive una **cartella**, non un singolo file: un file per partizione più `_SUCCESS`, il marcatore di scrittura completata. I file `.crc` sono checksum locali e li escludiamo dall'elenco.

In [ ]:
# Un file per partizione + _SUCCESS (marcatore di scrittura completata); i .crc sono checksum locali
print(sorted(f for f in os.listdir(f"{OUT}/ordini_parquet") if not f.endswith(".crc")))

### Punti chiave
* La stessa API `write.format(...)` serve per ogni formato; cambia solo il risultato su disco.
* Spark scrive in parallelo un file per partizione dentro una cartella.

## Step 3: Schema inferito vs memorizzato

Il CSV non contiene tipi: Spark deve leggere i dati per indovinarli (`inferSchema`, una lettura in più) oppure riceverli dall'utente. Senza inferenza tutte le colonne sono stringhe.
Parquet salva lo schema nel footer del file: la lettura è immediata e i tipi sono esatti.

In [ ]:
spark.read.option("header", True).csv(f"{OUT}/ordini_csv").printSchema()            # tutto stringa
spark.read.option("header", True).option("inferSchema", True).csv(f"{OUT}/ordini_csv").printSchema()
spark.read.parquet(f"{OUT}/ordini_parquet").printSchema()                            # tipi originali

## Step 4: Lettura colonnare e column pruning

Una query che usa **una sola colonna** (`avg(quantita)`) in Parquet legge solo quella colonna; in CSV/JSON deve analizzare ogni riga intera.
Per un confronto equo forniamo lo schema a CSV e JSON, così non pagano l'inferenza. La funzione `misura()` cronometra la query su ciascun formato.

In [ ]:
def misura(df, etichetta):
    t0 = time.time()
    df.select(F.avg("quantita")).collect()
    print(f"{etichetta:8s}: {time.time() - t0:.2f}s")

schema = ordini.schema   # schema fornito → niente inferenza, confronto più equo
misura(spark.read.schema(schema).option("header", True).csv(f"{OUT}/ordini_csv"), "csv")
misura(spark.read.schema(schema).json(f"{OUT}/ordini_json"), "json")
misura(spark.read.orc(f"{OUT}/ordini_orc"), "orc")
misura(spark.read.parquet(f"{OUT}/ordini_parquet"), "parquet")

## Step 5: Dentro un file Parquet

Con **PyArrow** leggiamo i metadati del primo file: numero di righe, row group e colonne, schema, e per ogni colonna codec di compressione, dimensione compressa e statistiche **min/max**. Sono queste statistiche a rendere possibile il salto dei blocchi.

In [ ]:
import pyarrow.parquet as pq

file0 = sorted(glob.glob(f"{OUT}/ordini_parquet/part-*.parquet"))[0]
meta = pq.ParquetFile(file0).metadata
print(f"File: {os.path.basename(file0)}\nrighe: {meta.num_rows:,}  row group: {meta.num_row_groups}  colonne: {meta.num_columns}")
print(pq.ParquetFile(file0).schema_arrow)

rg = meta.row_group(0)
print(f"\n{'colonna':12s} {'compressione':12s} {'byte':>10s}  min → max")
for i in range(rg.num_columns):
    col = rg.column(i)
    st = col.statistics
    print(f"{col.path_in_schema:12s} {col.compression:12s} {col.total_compressed_size:10,d}  {st.min} → {st.max}")

## Step 6: Predicate pushdown

Il filtro viene "spinto" nel lettore Parquet (`PushedFilters` nel piano): i row group le cui statistiche min/max escludono il valore cercato non vengono nemmeno letti.
`explain()` mostra il piano, `count()` esegue la query.

In [ ]:
pq_ordini = spark.read.parquet(f"{OUT}/ordini_parquet")
q = pq_ordini.where((F.col("ordine_id") < 1000) & (F.col("canale") == "app")).select("ordine_id", "data")
q.explain()
print("Righe trovate:", q.count())

### Punti chiave
* Column pruning: Parquet legge solo le colonne richieste.
* Predicate pushdown: le statistiche min/max permettono di saltare interi row group.
* Lo schema è nel file: nessuna inferenza e tipi esatti.

## Step 7: Partizionamento su disco (`partitionBy`)

Con `partitionBy` i dati vengono organizzati in sottocartelle `colonna=valore`. Le query che filtrano su quelle colonne leggono **solo le cartelle necessarie** (*partition pruning*).
Qui partizioniamo per `anno` e `mese` e stampiamo la struttura delle cartelle create.

In [ ]:
(ordini.withColumn("anno", F.year("data")).withColumn("mese", F.month("data"))
       .write.partitionBy("anno", "mese").mode("overwrite").parquet(f"{OUT}/ordini_partizionati"))

anni = sorted(d for d in os.listdir(f"{OUT}/ordini_partizionati") if d.startswith("anno="))
for d in anni:
    mesi = sorted((m for m in os.listdir(f"{OUT}/ordini_partizionati/{d}") if m.startswith("mese=")), key=lambda m: int(m[5:]))
    print(d, "→", mesi[:4], "...")

### Step 7b: Partition pruning nel piano

Filtrando su `anno` e `mese` il piano mostra `PartitionFilters`: Spark legge una sola cartella invece dell'intero dataset.

In [ ]:
part = spark.read.parquet(f"{OUT}/ordini_partizionati")
q = part.where("anno = 2025 AND mese = 3").groupBy("canale").count()
q.explain()        # PartitionFilters: [anno = 2025, mese = 3]
q.show()

### Step 7c: Con e senza partizionamento

Confrontiamo la stessa query sul dataset partizionato e su quello non partizionato (dove il filtro `year(data)` deve ispezionare i dati).

In [ ]:
# Confronto: la stessa query con e senza partizionamento
t0 = time.time(); part.where("anno = 2025 AND mese = 3").count(); t_part = time.time() - t0
t0 = time.time(); pq_ordini.where("year(data) = 2025 AND month(data) = 3").count(); t_full = time.time() - t0
print(f"partizionato: {t_part:.2f}s   non partizionato: {t_full:.2f}s")

## Step 8: Codec di compressione

Parquet supporta diversi codec: `snappy` (default, veloce), `zstd` (ottimo compromesso), `gzip` (compatto ma lento), `none`.
Scriviamo lo stesso DataFrame con ciascuno e confrontiamo spazio e tempo di scrittura.

In [ ]:
for codec in ["none", "snappy", "gzip", "zstd"]:
    path = f"{OUT}/codec_{codec}"
    t0 = time.time()
    ordini.write.option("compression", codec).mode("overwrite").parquet(path)
    print(f"{codec:7s}: {dimensione(path):6.1f} MB  scrittura {time.time() - t0:.2f}s")

## Step 9: Evoluzione dello schema

Nel tempo un dataset può acquisire nuove colonne. Scriviamo due versioni dei dati (la seconda con la colonna `email`) in sottocartelle `versione=N`.
Con l'opzione `mergeSchema` Spark unisce gli schemi dei diversi file; le righe vecchie avranno `NULL` nelle colonne nuove.

In [ ]:
v1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], "id INT, nome STRING")
v2 = spark.createDataFrame([(3, "Carla", "carla@example.com")], "id INT, nome STRING, email STRING")
v1.write.mode("overwrite").parquet(f"{OUT}/utenti/versione=1")
v2.write.mode("overwrite").parquet(f"{OUT}/utenti/versione=2")

spark.read.option("mergeSchema", True).parquet(f"{OUT}/utenti").orderBy("id").show()

### Punti chiave
* `partitionBy` accelera le query che filtrano sulle colonne di partizione, ma troppe partizioni generano troppi file piccoli.
* Il codec è un compromesso tra spazio e velocità.
* `mergeSchema` permette di leggere insieme file con schemi diversi.

## Step 10: SQL direttamente sui file

Spark SQL può interrogare un file senza registrare nulla, con la sintassi `` SELECT ... FROM parquet.`/percorso` ``.

In [ ]:
# Interrogare un file senza registrare nulla
spark.sql(f"SELECT canale, COUNT(*) AS n FROM parquet.`{OUT}/ordini_parquet` GROUP BY canale").show()

## Step 11: Tabelle gestite dal catalogo

`saveAsTable` scrive i dati come file Parquet nella cartella *warehouse* e registra la tabella nel catalogo (database `shop`). `DESCRIBE TABLE EXTENDED` ne mostra posizione e formato.
Senza un metastore (es. Hive) il catalogo vive in memoria: la definizione si perde a fine sessione, i file restano su disco.

In [ ]:
# Tabella gestita dal catalogo di Spark (file Parquet nella cartella "warehouse").
# Nota: senza un metastore (es. Hive) il catalogo vive in memoria e la definizione si perde a fine sessione.
# Il catalogo è in memoria ma i file restano su disco: li rimuoviamo per poter rieseguire la cella
shutil.rmtree(spark.conf.get("spark.sql.warehouse.dir").removeprefix("file:") + "/shop.db/ordini_negozio", ignore_errors=True)
spark.sql("CREATE DATABASE IF NOT EXISTS shop")
(ordini.where("canale = 'negozio'")
       .write.mode("overwrite").format("parquet").saveAsTable("shop.ordini_negozio"))
spark.sql("SHOW TABLES IN shop").show()
spark.sql("DESCRIBE TABLE EXTENDED shop.ordini_negozio").where("col_name IN ('Location','Provider','Type')").show(truncate=False)
spark.table("shop.ordini_negozio").count()

## Step 12: Chiudere la SparkSession

`spark.stop()` rilascia le risorse.

In [ ]:
spark.stop()

## Riepilogo

| Concetto | API / comando | A cosa serve |
|---|---|---|
| Scrittura multi-formato | `df.write.format(fmt).options(...).save()` | salvare in CSV, JSON, ORC, Parquet |
| Schema | `option("inferSchema")`, footer Parquet | tipi inferiti (CSV) o esatti (Parquet) |
| Column pruning | `spark.read.parquet(...).select(...)` | leggere solo le colonne necessarie |
| Metadati Parquet | `pyarrow.parquet.ParquetFile` | ispezionare row group, codec, statistiche |
| Predicate pushdown | `PushedFilters` in `explain()` | saltare row group tramite min/max |
| Partizionamento | `write.partitionBy(...)` | partition pruning su cartelle `col=valore` |
| Compressione | `option("compression", ...)` | scegliere tra snappy, zstd, gzip, none |
| Evoluzione schema | `option("mergeSchema", True)` | unire schemi di versioni diverse |
| SQL sui file / catalogo | ``parquet.`path` ``, `saveAsTable` | interrogare file e creare tabelle gestite |
| Chiusura | `spark.stop()` | rilasciare le risorse |